# 02 · Quality Analysis

Post-pipeline quality analysis using the processed fact table.
Run **after** `make run` to generate `data/processed/fct_transactions.csv`.


In [1]:
import sys
from pathlib import Path
ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

FACT = ROOT / 'data/processed/fct_transactions.csv'
df = pd.read_csv(FACT, low_memory=False)
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
print(f'Fact table: {len(df):,} rows × {len(df.columns)} cols')

Fact table: 4,947 rows × 26 cols


## 1. Pipeline cleaning summary

In [2]:
rejected = pd.read_csv(ROOT / 'data/interim/rejected_transactions.csv') \
          if (ROOT / 'data/interim/rejected_transactions.csv').exists() \
          else pd.DataFrame()
print(f'Fact rows    : {len(df):,}')
print(f'Rejected rows: {len(rejected):,}')
if not rejected.empty:
    print(rejected['_reject_reason'].value_counts().to_string())

Fact rows    : 4,947
Rejected rows: 49
_reject_reason
future_date       18
bad_unit_price    16
negative_price    15


## 2. Null rates in fact table

In [3]:
null_rates = (df.isna().mean() * 100).sort_values(ascending=False)
print('Null rates (%) — top 15:')
print(null_rates[null_rates > 0].head(15).round(2).to_string())

Null rates (%) — top 15:
holiday_name            100.00
date                    100.00
customer_tenure_days      4.20
SignupDate                4.20
Customer_Country          3.42
CustomerIsActive          3.42
CustomerID                3.42
CustomerSegment           3.42


## 3. Daily volume Z-score

In [4]:
daily = df.groupby(df['InvoiceDate'].dt.normalize()).size().rename('tx_count').sort_index()
roll_mean = daily.rolling(7, center=True, min_periods=1).mean()
roll_std  = daily.rolling(7, center=True, min_periods=1).std(ddof=1).fillna(1)
z = (daily - roll_mean) / roll_std.replace(0,1)

anomalous = z[z.abs() > 3]
print(f'Anomalous days (|z|>3): {len(anomalous)}')
if not anomalous.empty:
    print(anomalous.to_string())

Anomalous days (|z|>3): 0


## 4. Revenue breakdown

In [5]:
monthly = (df.assign(month=df['InvoiceDate'].dt.to_period('M'))
           .groupby('month')['LineTotal'].sum())
print('Monthly revenue (£):')
print(monthly.apply(lambda x: f'£{x:,.0f}').to_string())

Monthly revenue (£):
month
2023-01    £30,733
2023-02    £29,473
2023-03    £32,341
2023-04    £35,076
2023-05    £33,718
2023-06    £33,970
2023-07    £33,423
2023-08    £36,549
2023-09    £36,303
2023-10    £34,101
2023-11    £31,843
2023-12    £35,829
Freq: M


In [6]:
if 'CustomerSegment' in df.columns:
    seg = df.dropna(subset=['CustomerSegment']).groupby('CustomerSegment')['LineTotal'].agg(['sum','count','mean'])
    seg.columns = ['total_revenue','tx_count','avg_order_value']
    print(seg.sort_values('total_revenue', ascending=False).round(2).to_string())

                 total_revenue  tx_count  avg_order_value
CustomerSegment                                          
retail               133958.84      1640            81.68
online               133157.89      1606            82.91
wholesale            124649.77      1532            81.36


## 5. Orphan transaction analysis

In [7]:
if 'is_known_customer' in df.columns:
    orphan_rate = (~df['is_known_customer']).mean()
    print(f'Orphan transaction rate: {orphan_rate:.2%}')
    print(f'Orphan rows: {(~df["is_known_customer"]).sum():,}')
    print()
    print('Top countries with orphan transactions:')
    print(df[~df['is_known_customer']]['Country'].value_counts().head(10).to_string())

Orphan transaction rate: 3.42%
Orphan rows: 169

Top countries with orphan transactions:
Country
Finland           19
United Kingdom    13
Spain             13
Italy             12
Brazil            11
Denmark           10
Japan             10
Canada             9
Sweden             9
France             8


## 6. Holiday enrichment check

In [8]:
if 'is_holiday' in df.columns:
    print(f'Holiday-flagged transactions: {df["is_holiday"].sum():,}')
    print(f'(0 expected - pipeline ran with --no-api flag, no holiday data loaded)')

Holiday-flagged transactions: 0
(0 expected - pipeline ran with --no-api flag, no holiday data loaded)


## 7. Quality check summary

| Layer | Checks | Failures | Action |
|:------|:------:|:--------:|:-------|
| Validation (raw) | 14 | 4 | Pipeline continues -- cleaning resolves all 4 |
| DQ checks (clean) | 13 | 1 (info) | InvoiceNo non-unique -- expected (multi-line invoices) |
| Anomaly detection | 6 | 0 | All detectors within normal range |

**Overall: data is analytics-ready after the cleaning stage.**